# AI-Based Road Waterlogging Detection and Severity Estimation
**College Deep Learning Mini-Project**

This notebook demonstrates the end-to-end deep learning pipeline:
1. Dataset exploration and integrity checking
2. Preprocessing & synchronized data augmentation
3. Transfer learning model instantiation (MobileNetV2 backbone)
4. Training and evaluation metrics
5. Explainable visual localization (Grad-CAM)
6. Affected area calculation and severity categorization

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.model import create_model, get_device
from src.preprocessing import get_classification_transforms
from src.severity import SeverityEstimator
from src.segmentation import UNet, calculate_affected_area

device = get_device()
print(f"Executing on hardware: {device}")

## 1. Inspect Model Architecture (MobileNetV2 Baseline)
We use MobileNetV2 with an initial frozen feature backbone for transfer learning.

In [ ]:
model = create_model('mobilenet_v2', num_classes=2, pretrained=True, freeze_backbone_initial=True)
summary = model.get_parameter_summary()
print(f"Total parameters: {summary['total_parameters']:,}")
print(f"Trainable parameters (head only): {summary['trainable_parameters']:,}")

## 2. Test Severity Estimation Module
Severity is estimated transparently based on measured flood inundation percentage.

In [ ]:
estimator = SeverityEstimator()
test_cases = [0.0, 8.5, 24.3, 58.7]
for test_area in test_cases:
    res = estimator.estimate(affected_area_pct=test_area, waterlogging_detected=(test_area > 0))
    print(f"Area: {test_area:5.1f}% -> Severity: {res['severity_level']:<10} | Color: {res['color']} | {res['explanation']}")